# 01. Titanic — classification baseline

Классическая учебная задача (как на Kaggle Titanic): предсказать выживание.

**Стек:** pandas, sklearn  
**Цель:** EDA → признаки → train/test → logistic / random forest → метрики

💡 Все комментарии в коде — для разбора на созвоне.

In [ ]:
# cleanup: clearer train/test split notes
# импорты
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score

# путь к данным: .../data-notebooks/data/titanic.csv
DATA = Path("../data/titanic.csv")
print("file exists:", DATA.exists())

## 1. Загрузка и быстрый взгляд

💡 `Survived` — таргет (0/1). Остальное — признаки или мусор для модели (имя, билет).

In [ ]:
df = pd.read_csv(DATA)
print("shape:", df.shape)
print(df.columns.tolist())
df.head()

In [ ]:
# пропуски по столбцам — сколько NaN
print(df.isna().sum().sort_values(ascending=False).head(10))
print()
# доля выживших
print("survived rate:", round(df["Survived"].mean(), 3))
df.describe(include="all").T.head(12)

In [ ]:
# простой EDA: выживаемость по классу каюты и полу
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
df.groupby("Pclass")["Survived"].mean().plot(kind="bar", ax=axes[0], title="Survive rate by Pclass")
df.groupby("Sex")["Survived"].mean().plot(kind="bar", ax=axes[1], title="Survive rate by Sex")
plt.tight_layout()
plt.show()

## 2. Признаки

💡 Берём понятные поля. Имя/билет/cabin почти целиком не тащим в baseline (много мусора и уникальных значений).

In [ ]:
# числовые и категориальные признаки для первой модели
num_cols = ["Age", "SibSp", "Parch", "Fare"]
cat_cols = ["Pclass", "Sex", "Embarked"]

features = num_cols + cat_cols
target = "Survived"

# на всякий случай убираем строки без таргета (в этом файле их нет)
data = df.dropna(subset=[target]).copy()

X = data[features]
y = data[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=42,
    stratify=y,  # сохранить долю Survived
)

X_train.shape, X_test.shape, round(y_train.mean(), 3), round(y_test.mean(), 3)

## 3. Препроцессинг + Logistic Regression

💡 Пропуски в Age/Embarked заполняем. Категории → one-hot. Числа → scaler (для логрега удобно).

In [ ]:
# пайплайн: отдельно числа, отдельно категории, потом модель
preprocess = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),  # Age/Fare пропуски → медиана
            ("scaler", StandardScaler()),
        ]), num_cols),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),  # Embarked
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ]), cat_cols),
    ]
)

logreg = Pipeline([
    ("prep", preprocess),
    ("clf", LogisticRegression(max_iter=1000)),
])

logreg.fit(X_train, y_train)
pred = logreg.predict(X_test)
proba = logreg.predict_proba(X_test)[:, 1]

print("LogReg accuracy", round(accuracy_score(y_test, pred), 3))
print("LogReg ROC-AUC", round(roc_auc_score(y_test, proba), 3))
print(classification_report(y_test, pred, digits=3))

## 4. Сравнение с RandomForest

💡 На созвоне: «сначала baseline (logreg), потом сравнил с RF на том же split».

In [ ]:
# для дерева scaler не обязателен, но тот же imputer/onehot оставляем
rf_prep = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), num_cols),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ]), cat_cols),
    ]
)

rf = Pipeline([
    ("prep", rf_prep),
    ("clf", RandomForestClassifier(
        n_estimators=300,
        max_depth=6,
        random_state=42,
        n_jobs=-1,
    )),
])

rf.fit(X_train, y_train)
pred_rf = rf.predict(X_test)
proba_rf = rf.predict_proba(X_test)[:, 1]

print("RF accuracy", round(accuracy_score(y_test, pred_rf), 3))
print("RF ROC-AUC", round(roc_auc_score(y_test, proba_rf), 3))
print(classification_report(y_test, pred_rf, digits=3))

## 5. Вывод (для README / созвона)

1. Задача — classification (`Survived`).
2. Признаки — класс, пол, возраст, семья, тариф, порт.
3. Пропуски закрыл imputer, категории — one-hot.
4. Baseline LogReg, сравнение с RF.
5. Смотрю accuracy + ROC-AUC + report, не одну цифру.

Дальше можно: FamilySize = SibSp+Parch, Title из Name, CV вместо одного split.